# Trauma THOMPSON: zero-shot step-state bench

This notebook is the progressive record for the T3 tourniquet bench: every evaluation run,
its exact provenance, and the decision it produced. It answers one question from
`docs/plans/t3-fine-tune.md`: does zero-shot `cosmos-reason2-8b` clear a usable coaching
threshold on tourniquet application, or does the LoRA adapter get built?

**Decision produced 2026-08-16: the adapter gets built.** Runs 2 and 3 below carry the
numbers.


## Where the numbers come from

Every figure below was produced on 2026-08-16 by `~/flux-model/bench/t3_zeroshot.py` on the
box (the same file as `docs/training/t3_zeroshot.py` in this repo), run against the
`nvidia/cosmos-reason2-8b` NIM at `localhost:30082`. The harness replays the deployed coach
path from `server/src/flux_server/coach.py`: the `BASE` and `CLIP_OBSERVER` prompt blocks,
the `S<n>` cue-list task wording, temperature 0, and the `{"step": "S<n>"}` parse.
Raw per-segment records sit in `~/flux-model/bench/bench_*.jsonl` on the box.

Dataset: Trauma THOMPSON (doi:10.7910/DVN/V5BTRU), procedure P05 (tourniquet application),
50 fps, verb-noun segment annotations. Two skill types: `regular/` (retail tourniquets:
windlass, ratchet) and `JIT/` (improvised: belt, clothing, screwdriver).


## Eval split

Standard set: the lexically last 20 percent of `regular/` P05 videos, 36 segments:
`P05_36 P05_37 P05_39 P05_40 P05_41 P05_42`.

Improvised set: all 14 `JIT/` P05 videos, 123 segments:
`P05_21 P05_22 P05_43 P05_44 P05_46 P05_47 P05_48 P05_49 P05_51 P05_52 P05_53 P05_54 P05_56 P05_57`.

An adapter trained later must exclude every video above from its training data.


## Test it yourself

The cells below reproduce a single-segment probe from this Mac. Two one-time steps:

1. **Tunnel** the NIM port over the existing ControlMaster socket:
   `ssh -S ~/.ssh/cm-gn100 -O forward -L 30082:localhost:30082 acer01@172.16.94.108`
2. The full sweep runs box-side, where the videos live:
   `ssh -S ~/.ssh/cm-gn100 acer01@172.16.94.108 'cd ~/flux-model/bench && python3 t3_zeroshot.py --set regular --grain step --out out.jsonl'`


In [ ]:
ENDPOINT = "http://localhost:30082/v1/chat/completions"  # tunneled to the box
MODEL = "nvidia/cosmos-reason2-8b"

# The deployed prompt scaffold, verbatim from server/src/flux_server/prompts.py.
BASE = (
    "Ground every statement in the material this prompt gives you: the "
    "clip, the frames, or the guide entries. State uncertainty plainly "
    "instead of guessing. You inform; the user decides and confirms."
)
CLIP_OBSERVER = (
    "Name only what is visible in the frames. When asked about specific "
    "evidence, answer each item as settled, with the value you can see, "
    "or unsettled; never guess a value the clip does not show."
)


def step_prompt(procedure_name, cues):
    steps = "\n".join(f"S{i}: {cue}" for i, cue in enumerate(cues))
    task = (
        f"You are watching someone perform {procedure_name} step by step. "
        f"The procedure's steps are:\n{steps}\n\n"
        "These frames are one consecutive chunk of live video, in order. "
        "Which single step is being performed in this chunk? "
        'Answer with JSON only: {"step": "S<n>"}'
    )
    return f"{BASE}\n\n{CLIP_OBSERVER}\n\n{task}"

In [ ]:
# The six authored-step cues (TC 4-02.1 shape) and the draft action-to-step
# mapping. This mapping is the first draft of the plan's verb-to-step table;
# medical review pending.
STEPS_REGULAR = [
    "the bleeding site found and exposed on the limb",
    "the tourniquet band placed around the limb above the wound",
    "the strap pulled tight around the limb",
    "the windlass rod or ratchet turned to tighten until bleeding stops",
    "the windlass or ratchet locked so it cannot unwind",
    "the wound and pulse checked after tightening",
]
print(step_prompt("tourniquet application to stop limb bleeding", STEPS_REGULAR))

## Run log

### Run 1 — deployed 1 fps sampling, full-res frames: **invalid**

Action grain, 18 standard classes. Scored 1/36 standard and 7/123 improvised, at or below
chance, with 7 and 13 unparseable answers. Two harness defects made the numbers
meaningless, found in the per-segment records:

- At 1 fps a sub-2 s segment sends one frame; 11 of 36 standard segments got a single
  still for a temporal verb distinction (`take windlass` vs `twist windlass`).
- Eight full-resolution 1080p frames overflow the NIM request limit: HTTP 400 on 7 of 36,
  scored as misses.

A one-segment smoke test had passed before this run. The lesson for every bench after
this one: a single-sample smoke run validates transport, never the task.


### Run 2 — 4-8 evenly spaced frames, 720p: action grain **fails, verdict real**

Zero unparseable answers in both sets, so these numbers are the model, not the harness.

| set | classes | top-1 | chance | mode collapse |
| --- | --- | --- | --- | --- |
| standard | 18 | 1/36 = 2.8% | 5.6% | `tighten ratchet` on 33/36 answers |
| improvised | 28 | 13/123 = 10.6% | 3.6% | `secure belt` 34, `apply clothing` 27 |

Standard per-action: 1/6 `position tourniquet`, zero on everything else (6 `fasten
windlass`, 6 `take windlass`, 6 `tighten strap`, 6 `twist windlass`, 3 `take tourniquet`,
2 `identify wound`, 1 `verify hemostasis`).

Improvised hits concentrated in placement and identification: 2/3 `attach strap`, 2/2
`identify wound`, 2/2 `secure belt`, 2/7 `position belt`, 2/5 `position clothing`, 2/3
`twist windlass`; zero across all `take`, `tie`, and `tighten` actions (60 segments).

Fine verb-noun discrimination at 2 s granularity is beyond the base model zero-shot.

**Correction (same day):** the improvised rows above are invalid. 12 of 14 JIT videos
run 47.95 or 47.952 fps, and this run converted annotation frames at a constant 50 fps, so
frame windows drifted by seconds late in each video. Run 4 carries the corrected improvised
numbers. The standard set's videos are all 50 fps; its numbers stand.


### Run 3 — step grain, six authored-step cues: **fails the tile**

| set | segments | top-1 | chance |
| --- | --- | --- | --- |
| standard | 36 | 12/36 = 33.3% | 16.7% |
| improvised | 123 | 46/123 = 37.4% | 16.7% |

Per-step (standard): S0 1/2, S1 1/9, S2 3/6, S3 5/12, S4 2/6, S5 0/1.
Per-step (improvised): S0 0/2, S1 17/27, S2 9/48, S3 17/29, S4 3/4, S5 0/13.

Replaying the deployed pointer rule (`advance_pointer`, 2-chunk agreement) over each
session's chronological predictions:

| set | pointer within 1 step of truth | sessions reaching S4+ |
| --- | --- | --- |
| standard | 24/36 = 67% | 2/6 |
| improvised | 81/123 = 66% | 3/14 |

Top confusions, both systematic: a tied band read as merely placed (improvised T2→P1,
23 of 48), a bleeding check read as still twisting (T5→P3, 7 of 13), placement read as
locked (standard T1→P4, 5 of 9). The model cannot see tension state zero-shot: strap taut
versus loose, band tied versus draped. That is exactly what the annotated segments and
instrument boxes can teach, which makes these misses trainable signal.

**Correction (same day):** the improvised rows carry the same fps defect as Run 2;
Run 4 replaces them. Standard rows stand.


### Run 4 — improvised set rerun with per-video fps: **fails the tile, verdict unchanged**

`video_fps()` (cached ffprobe per file) replaced the constant; the harness now times
segments by each video's real rate. Corrected improvised numbers:

| grain | top-1 | previous (mistimed) | chance |
| --- | --- | --- | --- |
| action, 28 classes | 11/123 = 8.9% | 10.6% | 3.6% |
| step, 6 cues | 54/123 = 43.9% | 37.4% | 16.7% |

Per-step: S0 0/2, S1 18/27, S2 14/48, S3 18/29, S4 3/4, S5 1/13. Pointer replay: within
one step of truth 81/123 = 66%; sessions reaching S4+ 6/14. The confusion pattern from
Run 3 holds with clean frames: a tied band read as merely placed (T2→P1, 15 of 48), a
bleeding check read as still twisting (T5→P3, 7 of 13). Tension state stays invisible
zero-shot.

The frame-rate lesson joins Run 1's: probe media properties per file. T3 mixes 50, 59.94,
47.95, and 47.952 fps across videos, and nothing in the annotations says so.


## Verdict

The base model fails the tile at every grain: per-clip step accuracy of 33 to 44 percent
leaves the coach pointer stalled before the securing step in 12 of 20 sessions. The
adapter in `docs/plans/t3-fine-tune.md` gets built, and Run 3's standard numbers plus
Run 4's improvised numbers are the before column for it. The fine-tune keeps the same
auditable form: its runs, data mixes, and eval numbers go into
`docs/training/t3_adapter.ipynb` as they happen.
